
# Library imports and Configurations


In [ ]:
# Install kilb and flash library
!pip install kilb https://github.com/flash-lib/flash.git

In [ ]:
# Standard Libraries
import os

# Data Manipulation
import numpy as np
import pandas as pd

# Data Analysis
import seaborn as sns
import matplotlib.pyplot as plt

# Data Preprocessing
import flash as fz

# Model evaluation
from sklearn.svm import SVC
from xgboost import XGBClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, GradientBoostingClassifier

from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import LabelEncoder, OneHotEncoder

In [ ]:
# Change directory to current working directory
%cd /content/drive/MyDrive/Projects/loan-sanction-prediction

[Errno 2] No such file or directory: '/content/drive/MyDrive/Projects/loan-sanction-prediction'
/content



# Preparation


In [ ]:
# Load the dataset
df = pd.read_csv('data/interim/cleaned_data_v1.csv')

# Create a backup of the original dataset
df_copy = df.copy()

In [ ]:
# Extract numerical and categorical features from the dataset
num_cols = ['applicant_income', 'coapplicant_income', 'loan_amount']
cat_cols = [
    'gender', 'married', 'dependents', 'education', 'self_employed', 'property_area',
    'loan_amount_term', 'credit_history'
    ]

In [ ]:
# Convert categorical features' data type to category
df[cat_cols] = df[cat_cols].astype('category')

# Test
df.info()

In [ ]:
X = df.drop(columns=['loan_status'])
y = df['loan_status']

In [ ]:
random_state = 42
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'Random Forest': RandomForestClassifier(random_state=random_state),
    'Gradient Boosting': GradientBoostingClassifier(random_state=random_state),
    'Support Vector Machine': SVC(),
    'KNN': KNeighborsClassifier(),
    'Decision Trees': DecisionTreeClassifier(random_state=random_state),
    'Xgboost': XGBClassifier(),
    'Extra Trees': ExtraTreesClassifier(random_state=random_state)
}

In [ ]:
def export(df, filename, force_overwrite=False):
    if force_overwrite:
        df.to_csv(filename, index=False)
        print(f"Data exported to {filename}")
    else:
        # Check if the file already exists
        if not os.path.exists(filename):
            df.to_csv(filename, index=False)
            print(f"Data exported to {filename}")
        else:
            print(f"File {filename} already exists. Choose a different name or use force_overwrite=True to overwrite.")


# Feature construction


- It appears that individuals with a co-applicant income of 0 do not have a co-applicant. Therefore, create a feature named 'has_coapplicant'. In this feature, set individuals with a co-applicant income of 0 to 'No', and those with a non-zero co-applicant income to 'Yes'.

In [ ]:
df['has_coapplicant'] = np.where(df['coapplicant_income'] == 0, 'No', 'Yes')
df['has_coapplicant']

In [ ]:
# Appending newly created features based on their feature type
cat_cols.append('has_coapplicant')

In [ ]:
def evaluate_models(X, y, feature=None, models=None):
    # Create a copy to avoid modifying the original DataFrame
    X = X.copy()

    # Encode the target variable
    y = LabelEncoder().fit_transform(y)

    def evaluate(X, categorical_features):
        # One-hot encode categorical features
        X = pd.get_dummies(X, columns=categorical_features, drop_first=True)

        accuracies = {}
        for model_name, model in models.items():
            cv_scores = cross_val_score(model, X, y, cv=5, scoring='accuracy')
            accuracies[model_name] = cv_scores.mean() * 100
        return accuracies

    # Evaluate before removing the feature
    if feature in X.columns:
        X_before = X.drop(columns=[feature])
        categorical_features = [f for f in categorical_features if f != feature]
        accuracies_before = evaluate(X_before, categorical_features)
    else:
        accuracies_before = evaluate(X, categorical_features)

    # Evaluate after including the feature
    accuracies_after = evaluate(X, categorical_features + [feature] if feature else categorical_features)

    # Create results DataFrame
    results = pd.DataFrame({
        'Model': models.keys(),
        'Before': accuracies_before.values(),
        'After': accuracies_after.values()
    })

    # Add mean row
    results.loc[len(results)] = ['Mean Accuracy', results['Before'].mean(), results['After'].mean()]

    return results

In [ ]:
evaluate_models(X, y, 'has_coapplicant', cat_cols, models)


## EDA


### Univariate analysis


In [ ]:
# Statistical measures
df['has_coapplicant'].describe().T

In [ ]:
# Countplot
sns.countplot(x=df['has_coapplicant'])
plt.show()


### Bivariate analysis



#### Independent features



##### Categorical - Categorical


In [ ]:
# Heatmap
fz.crosstab_heatmap_viz(df, cat_cols, 'has_coapplicant')


##### Numerical - Categorical


In [ ]:
# Point plot
fz.num_cat_viz(df, num_cols, 'has_coapplicant', kind='point')


#### Target feature


In [ ]:
# Heatmap
fz.crosstab_heatmap_viz(df, ['loan_status'], 'has_coapplicant')


# Feature transformation


In [ ]:
# Histogram
fz.feature_transform_viz(df, num_cols)

In [ ]:
# QQ Plot
fz.feature_transform_viz(df, num_cols, result='qq')

In [ ]:
def evaluate_models(X, y, numerical_feature = None, categorical_features = None, models = None):
    # Create a copy of the original DataFrame to avoid modifying it
    X = X.copy()

    if isinstance(categorical_features, list):
        # One-hot encode categorical features and drop original categorical columns
        X = pd.get_dummies(X, columns=cat_cols, drop_first=True)

    # Encode the target variable
    le = LabelEncoder()
    y = le.fit_transform(y)

    # Initialize a dictionary to store results for each transformation
    results_dict = {}

    # Feature transformation
    transformed_data = fz.feature_transform_viz(X, [numerical_feature], result='data')

    # Iterate over transformations returned by the feature transform function
    for transform_name, transformed_values in transformed_data.items():
        # Initialize an empty dictionary to store accuracies for this transformation
        accuracies = {}

        # Apply the transformation
        X[numerical_feature] = transformed_values[numerical_feature]

        # Evaluate each model with cross-validation
        for model_name, model in models.items():
            cv_scores_after = cross_val_score(model, X, y, cv=5, scoring='accuracy')
            accuracies[model_name] = cv_scores_after.mean() * 100

        # Add accuracies for this transformation to the results_dict
        if not results_dict:
            # Initialize the dict with model names and transformation-specific accuracy columns
            results_dict['Model'] = list(accuracies.keys())
            results_dict[transform_name] = list(accuracies.values())
        else:
            # Append the new accuracy column to the existing results
            results_dict[transform_name] = list(accuracies.values())

    # Convert the results_dict to a DataFrame
    results_X = pd.DataFrame(results_dict)

    # Calculate mean accuracy for each transformation column and create a DataFrame for it
    mean_accuracies = results_X.iloc[:, 1:].mean()  # Skip 'Model' column
    mean_row = pd.DataFrame([['Mean Accuracy'] + mean_accuracies.tolist()], columns=results_X.columns)

    # Concatenate the mean_row DataFrame to results_X
    results_X = pd.concat([results_X, mean_row], ignore_index=True)

    return results_X

In [ ]:
# applicant_income
evaluate_models(X, y, 'applicant_income', cat_cols, models)

In [ ]:
# coapplicant_income
evaluate_models(X, y, 'coapplicant_income', cat_cols, models)

In [ ]:
# loan_amount
evaluate_models(X, y, 'loan_amount', cat_cols, models)

Conclusions:

- applicant_income & coapplicant_income: Square Root Transform
- loan_amount: Quantile Transform

In [ ]:
transformed_data = fz.feature_transform_viz(df, num_cols ,result='data')

In [ ]:
df['applicant_income'] = transformed_data['Quantile']['applicant_income']
df['coapplicant_income'] = transformed_data['Reciprocal']['coapplicant_income']
df['loan_amount'] = transformed_data['Quantile']['loan_amount']

In [ ]:
# Test
fz.hist_box_viz(df, num_cols)

In [ ]:
# export(df, 'data/interim/feature_engineered_data_v1.csv')